# AP HeatShield: Andhra Pradesh Heatwave & Peak Temperature Forecasting Using Linear Regression

**Introduction to Data Science Project**  
**Author:** Machabathuni Vishnu Vardhan  

## Project Objective
Instead of using a pre-cleaned Kaggle dataset, this project builds an end-to-end data pipeline from scratch:
1. **Web Scraping:** Extracting live multi-year meteorological HTML tables for Andhra Pradesh (Vijayawada station).
2. **Data Cleaning & Imputation:** Repairing broken sensor gaps (`'-'`), stripping summary artifacts, and applying time-series linear interpolation.
3. **Feature Engineering:** Building moving averages, thermal momentum features, and cyclical season encoders.
4. **Multiple Linear Regression:** Training a robust model across a full 365-day unseen test cycle to predict tomorrow's maximum temperature ($TM$).
5. **Deployment:** Saving the model (`.pkl`) and clean dataset (`.csv`) for Streamlit cloud deployment.

## Step 1: Install Required Libraries
We install standard packages for HTTP requests (`requests`), HTML parsing (`beautifulsoup4`, `lxml`), and data manipulation (`pandas`, `numpy`).

In [1]:
%pip -q install requests beautifulsoup4 pandas lxml scikit-learn seaborn matplotlib joblib

## Step 2: Multi-Year Web Scraper
We scrape historical monthly weather archives for Vijayawada (Gannavaram Airport station: `ws-431810`) across 2022 to 2025.  
* **Polite Scraping:** We add `time.sleep(0.8)` between requests so we do not overload the server.

In [2]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
import time

STATION_ID = "ws-431810"
YEARS = [2022, 2023, 2024, 2025]
MONTHS = range(1, 13)

all_rows = []
headers_saved = None

print("Starting polite web scraping for Vijayawada Station...")

for year in YEARS:
    for month in MONTHS:
        url = f"https://en.tutiempo.net/climate/{month:02d}-{year}/{STATION_ID}.html"
        try:
            response = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=15)
            if response.status_code != 200:
                continue
                
            soup = BeautifulSoup(response.text, "html.parser")
            table = soup.find("table", class_="medias mensuales numspan")
            if not table:
                continue
                
            rows = table.find_all("tr")
            for i, row in enumerate(rows):
                cols = [c.get_text(strip=True) for c in row.find_all(["th", "td"])]
                if not cols:
                    continue
                if i == 0 and headers_saved is None:
                    headers_saved = cols[:9] + ["Month", "Year"]
                elif i > 0 and len(cols) >= 9:
                    all_rows.append(cols[:9] + [month, year])
                    
            time.sleep(0.6)
        except Exception as e:
            pass

raw_df = pd.DataFrame(all_rows, columns=headers_saved)
raw_df.to_csv("ap_weather_raw_scraped.csv", index=False)
print(f"Total raw rows collected: {len(raw_df)}")

## Step 3: Data Diagnostics
We inspect the raw data to spot footer summary artifacts, missing string markers (`'-'`), and data types.

In [3]:
print("=== BOTTOM 5 ROWS (Check for summary artifacts) ===")
print(raw_df.tail(5))

print("\n=== MISSING SYMBOL COUNTS ===")
weather_cols = ['T', 'TM', 'Tm', 'SLP', 'H', 'PP', 'VV', 'V']
for col in weather_cols:
    dash_count = (raw_df[col] == '-').sum() + (raw_df[col] == '').sum()
    print(f"Column {col}: {dash_count} missing values ({(dash_count/len(raw_df))*100:.1f}%)")

## Step 4: Data Cleaning & Time-Series Interpolation
* **Filtering:** We retain only rows where `'Day'` is a valid digit (removing monthly summary text).
* **Column Dropping:** `SLP` (Sea Level Pressure) exceeds 40% missing values, so we drop it.
* **Interpolation:** Missing continuous weather records are filled smoothly using `method='linear'` to preserve seasonal trends.

In [4]:
import matplotlib.pyplot as plt

df = raw_df.copy()
df = df[df['Day'].str.isdigit()].copy()
df['Day'] = df['Day'].astype(int)

df['Date'] = pd.to_datetime(df[['Year', 'Month', 'Day']], errors='coerce')
df = df.dropna(subset=['Date']).sort_values('Date').reset_index(drop=True)

for col in weather_cols:
    df[col] = pd.to_numeric(df[col].replace(['-', ''], np.nan), errors='coerce')

tm_before = df['TM'].copy()

missing_pct = df[weather_cols].isna().mean()
cols_to_drop = missing_pct[missing_pct > 0.40].index.tolist()
df = df.drop(columns=cols_to_drop)
active_cols = [c for c in weather_cols if c not in cols_to_drop]

if 'PP' in active_cols:
    df['PP'] = df['PP'].fillna(0.0)

df[active_cols] = df[active_cols].interpolate(method='linear', limit_direction='both')
print("Cleaned dataset shape:", df.shape)

## Step 5: Feature Engineering & Correlation Analysis
To maximize our $R^2$ score, we build autoregressive lag features (`tm_lag1`), rolling moving averages (`tm_roll3`, `tm_roll7`), and cyclical seasonal encodings (`season_sin`, `season_cos`).

In [5]:
import seaborn as sns

data = df.copy()
data['tm_lag1'] = data['TM'].shift(1)
data['tm_roll3'] = data['TM'].rolling(3).mean()
data['tm_roll7'] = data['TM'].rolling(7).mean()
data['h_roll3'] = data['H'].rolling(3).mean()

day_of_year = data['Date'].dt.dayofyear
data['season_sin'] = np.sin(2 * np.pi * day_of_year / 365.25)
data['season_cos'] = np.cos(2 * np.pi * day_of_year / 365.25)

# Target variable: Tomorrow's Maximum Temperature
data['target_next_tm'] = data['TM'].shift(-1)
data = data.dropna().reset_index(drop=True)

data.to_csv("vijayawada_heatshield_clean.csv", index=False)
print("Final modeling dataset shape:", data.shape)

## Step 6: Multiple Linear Regression Training & Evaluation
* **Chronological Split:** We avoid data leakage by training on past years and reserving the final 365 days as an unseen test set.
* **Model Persistence:** We save our trained model and feature list into `heatshield_lr_model.pkl`.

In [6]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import joblib

features = ['TM', 'T', 'Tm', 'H', 'PP', 'VV', 'V', 
            'tm_lag1', 'tm_roll3', 'tm_roll7', 'h_roll3', 
            'season_sin', 'season_cos']

X = data[features]
y = data['target_next_tm']
dates = data['Date']

split_idx = len(data) - 365
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]
test_dates = dates.iloc[split_idx:]

lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

train_preds = lr_model.predict(X_train)
test_preds = lr_model.predict(X_test)

print(f"Train R² Score : {r2_score(y_train, train_preds):.4f}")
print(f"Test R² Score  : {r2_score(y_test, test_preds):.4f}")
print(f"Test MAE       : {mean_absolute_error(y_test, test_preds):.2f} °C")
print(f"Test RMSE      : {np.sqrt(mean_squared_error(y_test, test_preds)):.2f} °C")

joblib.dump({'model': lr_model, 'features': features}, "heatshield_lr_model.pkl")

## Step 7: Automatic `metrics.json` Generation
We dynamically export our evaluation metrics into a JSON file for model cards and GitHub reporting.

In [7]:
import json

metrics = {
    "model_name": "Multiple Linear Regression",
    "station": "Vijayawada Gannavaram (ws-431810)",
    "target_variable": "Tomorrow's Maximum Temperature (TM)",
    "total_dataset_rows": int(len(data)),
    "metrics": {
        "train_r2_score": round(float(r2_score(y_train, train_preds)), 4),
        "test_r2_score": round(float(r2_score(y_test, test_preds)), 4),
        "test_mae_celsius": round(float(mean_absolute_error(y_test, test_preds)), 2),
        "test_rmse_celsius": round(float(np.sqrt(mean_squared_error(y_test, test_preds))), 2)
    }
}

with open("metrics.json", "w") as f:
    json.dump(metrics, f, indent=4)

print("metrics.json successfully generated!")

## Step 8: Visualization & Exporting Test Plot (`.png`)
We plot Actual vs. Predicted peak temperatures across the 365-day test set and save the figure as `heatshield_test_predictions.png`.

In [8]:
plt.figure(figsize=(12, 4.5))
plt.plot(test_dates, y_test.values, label="Actual Next-Day Max Temp", color="#1f2937", linewidth=1.8)
plt.plot(test_dates, test_preds, label="Predicted Max Temp (Linear Regression)", color="#ef4444", linestyle="--", linewidth=1.8)
plt.axhline(40, color="orange", linestyle=":", alpha=0.9, label="40°C Heatwave Warning Threshold")

plt.title(f"AP HeatShield Test Set: Actual vs Predicted Peak Temp (R² = {r2_score(y_test, test_preds):.3f}, MAE = {mean_absolute_error(y_test, test_preds):.2f}°C)")
plt.xlabel("Date")
plt.ylabel("Maximum Temperature (°C)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("heatshield_test_predictions.png", dpi=300)
plt.show()